# 4 - IP Build RCAs from Pre-Processed NetMap Reaches and DEM

Use this tool to build reach contributing area (RCAs) from a NetMap stream reaches and a digital elevation model (DEM). This tool is designed to replace the STARS "Generate Cost RCAs" tool which was built using Python 2 and designed for use in ArcMap.

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.


## Required Inputs:

- A DEM, preferably the same source that was used to create the NetMap reaches. For the Yukon-Kuskokwim IP project, this is the 5m IFSAR Digital Surface Model (DSM) that can be downloaded from various repositories online. See the Data.gov catalog entry for metadata (https://catalog.data.gov/dataset/5-meter-alaska-digital-elevation-models-dems-usgs-national-map-3dep-downloadable-data-collectio) and the USGS National Map online application for a download option (https://apps.nationalmap.gov/downloader/).

- NetMap synthetic stream reaches, clipped to HUC8 extent and pruned to keep only those reaches with greater than 5km catchment area. These reaches should be pre-processed using the "Prune Netmap Reaches to 5km and Clip by HUC8" script included in this toolbox. (Attempting to process a full size NetMap product with a 5m DEM will likely crash the program.)

- A unique attribute field in the pre-processed NetMap synthetic stream reach feature layer. The values in this field will become the unique RCA IDs.

- A feature class name for the RCA polygon output.

- A system directory to use as a temporary file location. It is recommended that this be a short file path on an internal hard drive (eg, "C:/data").


## Geoprocessing Output:

- An RCA polygon feature class written to the default project geodatabase.

## Processing Steps:

1. Convert the pre-processed reaches to a raster, using the same cell size and extent as the input DEM.
2. Create a raster object from the rasterized streamlines.
3. Run a series of Raster Calculator expressions. (These calculations essentially create a cost surface where ridgeline landscape positions become very expensive compared to valley and hillslope positions. This will constrain the RCA polygons and prevent them from crossing ridgetops into adjacent drainages.)
4. Run the Cost Allocation using the rasterized stream reaches and the final cost surface.
5. Build a raster attribute table for the the cost allocation output.
6. Convert the cost allocation output to polygon, and dissolve any multi-part polygons.
7. Add a field for the RCA ID, then calculate the RCA ID to equal the "gridcode" attribute from the original rasterized reaches.
8. Delete all intermediary rasters.

### Code starts here:

#### Setup

Import modules and reset environments to default. This should set the ArcPro project geodatabase as the workspace/scratch environment, just in case it was set otherwise. Additionally, prevent the addition of intermediary outputs to the ArcPro project map. Some tools may not run if their target is open in the map display.

In [134]:
import arcpy
import os

arcpy.ResetEnvironments()
arcpy.env.addOutputsToMap = False

The Spatial Analyst extension is required for this script to work. Check for extension, and if its available check it out for use. If it is unavailable or cannot be checked out, throw a license error and provide an error message.

In [ ]:
print("Spatial Analyst license is required.")
arcpy.AddMessage("Spatial Analyst license is required.")
print(arcpy.GetMessages())

class LicenseError(Exception):
    pass

try:
    if arcpy.CheckExtension("Spatial") == "Available":
        arcpy.CheckOutExtension("Spatial")
        print("Good news...Spatial Analyst license is available!")
        arcpy.AddMessage("Good news...Spatial Analyst license is available!")
        print(arcpy.GetMessages())
    else:
        # raise a custom exception
        raise LicenseError

except LicenseError:
    print("Bad news...Spatial Analyst license is unavailable.")
    arcpy.AddMessage("Bad news...Spatial Analyst license is unavailable.")
    print(arcpy.GetMessages())
except arcpy.ExecuteError:
    print(arcpy.GetMessages())

User provides filepaths to the DEM, pre-processed stream reaches, and output RCA polygon name, and a system folder temporary workspace. The unique stream reach ID field is also provided.

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [70]:
DEM = "F:/GIS/IP_DATA/IFSAR_MOSAICS/DEM_5m_19080305_clip.tif"

InReach = "F:/GIS/IP/Yukon_Tanana.gdb/reach_Tanana_5km_19080305_HUC8"

reach_field = "RCA_ID"

OutRCA = "RCA_19080305_HUC8"

WKSpace = "C:/data"

arcpy.env.scratchWorkspace = WKSpace

In [ ]:
#DEM = arcpy.GetParameterAsText(0)

#InReach = arcpy.GetParameterAsText(1)

#reach_field = arcpy.GetParameterAsText(2)

#OutRCA = arcpy.GetParameterAsText(3)

###WKSpace = arcpy.GetParameterAsText(4)

#arcpy.env.scratchWorkspace = WKSpace

Convert the pruned NetMap reaches to a raster, using the same cell size as the DEM. Also, use the DEM as the snap raster.

In [71]:
DEM_ras = arcpy.Raster(DEM)
size = DEM_ras.meanCellWidth
size = int(size)
arcpy.env.snapRaster = DEM_ras
arcpy.conversion.PolylineToRaster(InReach, reach_field, "ReachRas", "", "", size, "")

<Result 'F:\\GIS\\IP\\IP.gdb\\ReachRas'>

Create a raster object from the rasterized streamlines to allow us to run a series of Raster Calculator expressions.

In [72]:
Reach_ras = arcpy.Raster("ReachRas")

Focal Mean equation using the annulus shape around each raster cell. (Annulus size parameters used here are copied directly from the STARS tool.)

In [73]:
nbr = arcpy.sa.NbrAnnulus(7, 12, "CELL")
Ann_Raw = arcpy.sa.Int((DEM_ras - (arcpy.sa.FocalStatistics(DEM_ras, nbr, "MEAN", ""))) + 0.5)

Weight the ridgelines using a conditional statement. If the difference between the DEM elevation and the annulus focal mean was greater than 0, that means the pixel is probably on a convexity (ie, surrounded by lower elevation areas). In the convex condition, increase the elevation exponentionally to provide a heavy cost for traversing that pixel. In non-convex conditions, assign a value of 1. 

In [74]:
Ann_Cost = arcpy.sa.Con(Ann_Raw > 0, arcpy.sa.Power(Ann_Raw, 1.9), 1)

Calculate DEM slope, flow direction, and flow accumulation.

In [75]:
slp = arcpy.sa.Slope(DEM_ras, "DEGREE")

flowdir = arcpy.sa.FlowDirection(DEM_ras, "FORCE", "")

flowacc = arcpy.sa.FlowAccumulation(flowdir)

Run focal statistics on the flow accumulation layer to determine the maximum flow accumulation in a 6 cell radius. 

In [76]:
nbr2 = arcpy.sa.NbrCircle(6, "CELL")
flowmax = arcpy.sa.FocalStatistics(flowacc, nbr2, "MAXIMUM", "")

Apply the hydrologic weights equation copied directly from the STARS tool. In this equation, where the maximum flow accumulation is greater than 1, replace those values with a non-zero topographic wetness index. Where the maximum flow is less than one, or slope is zero, replace those values with 1.

This step should take all areas with very low flow accumulation (ie ridgelines) and assign their value as 1. All other areas will have a hydrologic weight calculated.

In [77]:
flow_wt = arcpy.sa.Con(flowmax > 1, arcpy.sa.Con(slp > 1, (1.0 / flowmax * arcpy.sa.Tan(slp / 57.2957)), 1), 1)

Calculate a cost surface using the flow weight. In this equation, where the DEM cost surface is less than 100 million (ie, all pixels), multiply the cost surface by the flow weight pixels with value greater than or equal to 1. If the flow weight pixels have value less than or equal to 1, multiply the cost surface by 0.1.

In areas of high cost identified from the DEM (ie ridgelines), we are multiplying them by the hydrologic weight of 1 (the default hydro weight value for areas of low flow). The high cost on those ridgelines will therefore be unchanged.

In all other low cost areas identified from the DEM (ie, non-ridgelines), multiplying them by the hydrologic weight should produce somewhat higher values. (The intention of this tool is a scaling step that will make the following cost allocation tool perform correctly.)

In [78]:
cost_surf = arcpy.sa.Con(Ann_Cost < 100000000, Ann_Cost * arcpy.sa.Con(flow_wt >= 1, flow_wt, 0.1), 100000000)

Run the Cost Allocation using the rasterized stream reaches and the final cost surface to create the base RCA raster.

In [79]:
RCA_ras = arcpy.sa.CostAllocation(Reach_ras, cost_surf)

Build a raster attribute table in the RCA raster.

In [80]:
arcpy.management.BuildRasterAttributeTable(RCA_ras, "Overwrite")

<Result 'C:\\data\\CostAll_ReachRa1.tif'>

Convert the RCA raster to polygon, and dissolve any multipart polygons.

In [81]:
arcpy.conversion.RasterToPolygon(RCA_ras, "OutRCA_temp", "SIMPLIFY", "Value")
arcpy.management.Dissolve("OutRCA_temp", OutRCA, "GRIDCODE", "", "MULTI_PART")

<Result 'F:\\GIS\\IP\\IP.gdb\\RCA_19080305_HUC8'>

Add a field for the RCA ID, and calculate that field.

In [82]:
arcpy.management.AddField(OutRCA, "RCA_ID", "LONG", "", "", "", "")
arcpy.management.CalculateField(OutRCA, "RCA_ID", "!GRIDCODE!")

<Result 'F:\\GIS\\IP\\IP.gdb\\RCA_19080305_HUC8'>

In [83]:
arcpy.management.Delete(Ann_Raw)

<Result 'true'>

In [ ]:
arcpy.management.Delete(Ann_Raw)
arcpy.management.Delete(flowdir)
arcpy.management.Delete(flowacc)
arcpy.management.Delete(flowmax)
arcpy.management.Delete(slp)
arcpy.management.Delete(flow_wt)
arcpy.management.Delete(Reach_ras)
arcpy.management.Delete(cost_surf)
arcpy.management.Delete(RCA_ras)
try:
    arcpy.management.Delete("OutRCA_temp")
except:
    desc = arcpy.Describe("OutRCA_temp")
    arcpy.management.Delete(desc.path)
except:
    print("Deletion error.... check project GDB and scratch workspace for remaining files...")